# 🧪 Unit III: Classification Analysis (Churn Prediction)

## 1. Executive Summary
In this unit, we predict whether a customer will **Return** (Loyal) or is a **One-time Buyer** (Churn Risk).

**Objectives:**
1.  **Algorithms**: Naive Bayes, Decision Tree, SVM, KNN.
2.  **Evaluation Criteria**: Accuracy, Precision, Recall, F1-Score, AUC-ROC, and Log Loss.
3.  **Outcome**: Identify the best model for targeting retention campaigns.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Add src to path
sys.path.append(os.path.abspath(os.path.join('..')))
from src.models.classification import CustomerClassifier

# Load processed RFM data
rfm = pd.read_csv('../data/processed/rfm_customer_data.csv')
print("✅ RFM Data Loaded")

## 2. Data Preparation
We define the target variable `IsReturn` as 1 if `Frequency > 1`, else 0.
We use `Recency` and `Monetary` as features.

In [ ]:
clf_model = CustomerClassifier()
X_train, X_test, y_train, y_test = clf_model.prepare_data(rfm)

print(f"Training set Class Balance:\n{y_train.value_counts(normalize=True)}")

## 3. Model Training & Evaluation
We train all 4 classifiers and gather their metrics.

In [ ]:
results = clf_model.train_evaluate_all(X_train, X_test, y_train, y_test)
results_df = pd.DataFrame(results).T
print("Model Comparison Table:")
print(results_df)

## 4. Performance Visualization
Visualizing Accuracy and F1-Score to pick the winner.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
results_df['Accuracy'].plot(kind='bar', ax=ax[0], color='skyblue', alpha=0.8)
ax[0].set_title('Accuracy Comparison')
ax[0].set_ylim(0, 1)

# F1 Score
results_df['F1'].plot(kind='bar', ax=ax[1], color='lightgreen', alpha=0.8)
ax[1].set_title('F1-Score Comparison')
ax[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 5. ROC Curve Analysis
Which model best discriminates between classes?

In [ ]:
from sklearn.metrics import roc_curve

plt.figure(figsize=(10, 6))
for name, model in clf_model.models.items():
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        plt.plot(fpr, tpr, label=f"{name} (AUC={results[name].get('AUC_ROC', 0):.2f})")

plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.show()

**📝 Insight:**
- **AUC-ROC**: Values closer to 1.0 indicate excellent performance.
- **Precision vs Recall**: If we care more about finding *all* return customers, we look at Recall. If we want to be sure our 'Loaylty' coupons aren't wasted, we look at Precision.